In [1]:
import os

# Set the environment variable to a higher timeout (e.g., 2 seconds)
os.environ['PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT'] = '2.0'

In [2]:
import mne
import numpy as np
import scipy.signal as signal
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import simps
from tqdm.notebook import tqdm
import os
from scipy.stats import skew, kurtosis, entropy
import scipy.stats as stats
from scipy.signal import find_peaks
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import spearmanr

In [3]:
# create folder to store result of Baseline correction aögorithm if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/BCA"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [4]:
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")
df_filtered = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]
df_skill = df_skill[["Participant", "SkillScore"]]
df_skill = df_skill.drop_duplicates()
df_filtered

,Participant,Algorithm,SkillScore,EEG,CrossEEG
0,1,IsPrime,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1,1,SiebDesEratosthenes,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
2,1,IsAnagram,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
3,1,RemoveDoubleChar,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
4,1,BinToDecimal,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
...,...,...,...,...,...
1067,71,DumpSorting,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1068,71,BinomialCoefficient,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1069,71,IsAnagram,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1070,71,ArrayAverage,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...


### Skill level assignment

- less than 0.33 quantile --> Novice  
- more than 0.66 quantile --> Expert  
- else --> Intermediate  

In [5]:
quantile = df_skill["SkillScore"].quantile([0.33, 0.66])
lower = quantile[0.33]
upper = quantile[0.66]

df_skill['SkillLevel'] = np.select([df_skill['SkillScore'] < lower, (df_skill['SkillScore'] >= lower) & (df_skill['SkillScore'] <= upper), df_skill['SkillScore'] > upper],
                                 ['Novice', 'Intermediate', 'Expert'],
                                 default='Intermediate')
df_skilled = df_skill.copy()
df_skilled

,Participant,SkillScore,SkillLevel
0,1,0.331385,Intermediate
32,2,0.379187,Expert
64,3,0.311264,Intermediate
80,4,0.424727,Expert
112,5,0.313031,Intermediate
144,6,0.315932,Intermediate
173,7,0.420873,Expert
205,10,0.350392,Expert
237,11,0.178206,Novice
261,12,0.309233,Intermediate


In [6]:
df_skill.groupby('SkillLevel')['Participant'].count()

SkillLevel
Expert          13
Intermediate    12
Novice          12
Name: Participant, dtype: int64

### Functions to calculate Brainwaves

In [7]:
def _to_decibel(spec):
    return 10 * np.log10(spec)

def get_spectrum(data, sampling_rate, method='welch', decibel=False, resolution='auto'):
    """
    Calculate amplitude or power spectrum

    data: Should be of shape (n_channels, n_samples)
    sampling_rate: Sampling rate... (float)
    method:
        * welch for power spectrum using Welch's method (recommended)
        * ft for simple Fourier transform (amplitude spectrum)
        * ps for power spectrum using a simple fourier transform
    decibel: Convert spectrum to decibel (bool)
    """

    axis = -1

    n_channels, n_samples = data.shape

    if resolution == 'auto':
        n_frequencies = n_samples
    elif isinstance(resolution, (int, float)):
        n_frequencies = np.round(sampling_rate / resolution).astype('int')
    else:
        raise ValueError('\'{}\''.format(resolution))

    # Spectrum
    if method in ['ft', 'ps']:
        # Using FFT
        # Get (complex) spectrum
        spec = np.fft.fft(data, n=n_frequencies, axis=axis)
        freq = np.fft.fftfreq(n_frequencies) * sampling_rate

        # Convert to real positive-sided spectrum
        spec = np.abs(spec)
        nyquist = 0.5 * sampling_rate
        is_positive = np.logical_or(np.logical_and(freq >= 0, freq <= nyquist), freq == -nyquist)
        n_pos = np.sum(is_positive)
        is_positive = np.repeat(is_positive[np.newaxis], n_channels, axis=0)
        spec = np.reshape(spec[is_positive], (n_channels, n_pos))
        freq = np.abs(freq[is_positive])
        is_double = np.logical_and(freq > 0, freq < nyquist)
        is_double = np.repeat(is_double[np.newaxis], n_channels, axis=0)
        spec[is_double] = 2 * spec[is_double]

        if method in ['ps']:
            # Get power spectral density
            spec = (1 / (sampling_rate * n_frequencies)) * spec ** 2

            # Convert to decibel if required
            if decibel:
                spec = _to_decibel(spec)
    elif method in ['welch', 'welch_db']:
        # Using Welch method
        freq, spec = signal.welch(data, sampling_rate, nperseg=n_frequencies, detrend='constant', axis=axis)

        # Convert to decibel if required
        if decibel:
            spec = _to_decibel(spec)
    else:
        raise RuntimeError('Unknown method \'{}\''.format(method))

    return spec, freq

In [30]:
#to extract the power within a specified frequency band from a power spectrum, 
#and  optionally normalizes the result if relative is set to True

def bandpower(spec, freq, freqband, relative=False):
    """
    Get band power within specified frequency band
    Alternatively: https://raphaelvallat.com/bandpower.html
    """

    spec = np.asarray(spec)
    freq = np.asarray(freq)
    freqband = np.asarray(freqband)

    if spec.ndim != 1:
        raise ValueError('Input \'spec\' bad: {}'.format(spec.shape))

    if freqband.ndim != 1 and freqband.shape[-1] != 2:
        raise ValueError('Input \'freqband\' bad: {}'.format(freqband.shape))

    # Frequency resolution
    step_freq = freq[1] - freq[0]

    # Find closest indices of band in frequency vector
    is_in_freqband = np.logical_and(freq >= np.min(freqband), freq <= np.max(freqband))

    # Integral approximation of the spectrum using Simpson's rule
    bp = simps(spec[is_in_freqband], dx=step_freq)

    if relative:
        bp = bp / simps(spec, dx=step_freq)

    return bp

In [62]:
def baseline_correction(raw_data, baseline_data):
    
    if baseline_data.shape > raw_data.shape:
        #Make sure that the baseline signal have same length as eeg raw signal
        baseline_data = baseline_data[:, :raw_data.shape[1]]
    else:
        raw_data = raw_data[:, :baseline_data.shape[1]]
    #Baseline Correction 
    eeg = np.subtract(raw_data, baseline_data)
    return eeg

In [63]:
def brainwaves(eeg_path, cross_eeg_path, sampling_rate=500):
    # read in eeg file
    eeg_data = mne.io.read_raw_fif(eeg_path, preload=True, verbose='ERROR')

    # get raw channel data and do mean average referencing
    eeg_data_raw = eeg_data.get_data()
    #eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

    # read in eeg file
    baseline_eeg_data = mne.io.read_raw_fif(cross_eeg_path, preload=True, verbose='ERROR')

    # get raw channel data and do mean average referencing
    eeg_data_baseline = baseline_eeg_data.get_data()
    #eeg_data_baseline_ref = eeg_data_baseline - np.mean(eeg_data_baseline, axis=0)

    eeg_corrected = baseline_correction(eeg_data_raw, eeg_data_baseline)
    # extract channel names of eeg_data
    channel_names = list(eeg_data.to_data_frame().columns[1:])

    # create mock events for cutting eeg data (number, len, id)
    events = np.array([(0, 0, 1)])

    # create temporal eeg raw for cutting data into epochs
    tmp_raw = mne.io.RawArray(eeg_corrected, eeg_data.info, verbose='ERROR')

    # Considered min. duration of the Participant's code comprehension
    duration = 4

    # create epochs which have a duration second window and operate on event id 1
    epochs = mne.Epochs(tmp_raw, events, event_id=1, tmin=0, tmax=duration,baseline=None, preload=True, verbose='ERROR')
    #first_epoch = epochs[0]
    alpha = np.array([])
    beta = np.array([])
    gamma = np.array([])
    theta = np.array([])
    range_4_to_50 = np.array([])

    for epoch_data_raw in epochs:
    # perform power spectrum analysis on eeg data
        spectrum, frequency = get_spectrum(epoch_data_raw, sampling_rate, method='welch', decibel=False,
                                        resolution='auto')

        alpha_current = np.array([])
        beta_current = np.array([])
        gamma_current = np.array([])
        theta_current = np.array([])
        range_4_to_50_current = np.array([])

        for channel in channel_names:
            alpha_current = np.append(alpha_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [8.0, 13.0],
                                                relative=True))
            beta_current = np.append(beta_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [13.0, 30.0],
                                            relative=True))
            gamma_current = np.append(gamma_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [30.0, 50.0],
                                                relative=True))
            theta_current = np.append(theta_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [4.0, 8.0],
                                                relative=True))
            range_4_to_50_current = np.append(range_4_to_50_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [4.0, 50.0],
                                                relative=True))

        alpha = np.append(alpha, alpha_current)
        beta = np.append(beta, beta_current)
        gamma = np.append(gamma, gamma_current)
        theta = np.append(theta, theta_current)
        range_4_to_50 = np.append(range_4_to_50, range_4_to_50_current)

    return alpha, beta, gamma, theta, range_4_to_50

### Brainwaves calculation

In [64]:
# read in montage and calculate the electrode positions
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")

df_eeg_data = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]

# sampling rate of the EEG data
sampling_rate = 500

# create Mental Workload column
df_brain_waves = pd.DataFrame(columns=["Participant", "SkillScore", "Algorithm", "Alpha", "Beta", "Gamma", "Theta", "Range4to50Hz"])

# iterate over each row anc calculate barin waves for the task
for idx in tqdm(range(len(df_eeg_data))):
    participant = df_eeg_data.iloc[idx]["Participant"]

    algorithm = df_eeg_data.iloc[idx]["Algorithm"]
    skill_score = df_eeg_data.iloc[idx]["SkillScore"]
    eeg_path = df_eeg_data.iloc[idx]["EEG"]
    cross_eeg_path = df_eeg_data.iloc[idx]["CrossEEG"]

    

    alpha, beta, gamma, theta, range4to50 = brainwaves(eeg_path, cross_eeg_path)
    

    #baseline_alpha, baseline_beta, baseline_gamma, baseline_theta, baseline_range4to50 = brainwaves(cross_eeg_path)
    alpha = alpha.reshape(64, 1)
    beta = beta.reshape(64, 1)
    gamma = gamma.reshape(64, 1)
    theta = theta.reshape(64, 1)
    range4to50 = range4to50.reshape(64,1)
    
    #baseline_alpha = baseline_alpha.reshape(64, 1)
    #baseline_beta = baseline_beta.reshape(64, 1)
    #baseline_gamma = baseline_gamma.reshape(64, 1)
    #baseline_theta = baseline_theta.reshape(64, 1)
    #baseline_range4to50 = baseline_range4to50.reshape(64,1)


    df_brain_waves.loc[len(df_brain_waves)] = [participant, skill_score, algorithm, alpha, beta, gamma, theta, range4to50]
 
df_brain_waves.to_csv(result_path+"/BrainWaves.csv")


  0%|          | 0/1072 [00:00<?, ?it/s]

In [24]:
df_brain_waves

,Participant,SkillScore,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz,BaselineAlpha,BaselineBeta,BaselineGamma,BaselineTheta,BaselineRange4to50Hz
0,1,0.331385,IsPrime,"[[0.17390608910662744], [0.1595872503199541], ...","[[0.09869159888676664], [0.17572296368665366],...","[[0.09502044179307881], [0.07892026789143232],...","[[0.09961805208244852], [0.10068577398697535],...","[[0.48627747872221094], [0.537527069147148], [...","[[0.7738523140576222], [0.7267182143147234], [...","[[0.03348518144395066], [0.05105540668186075],...","[[0.024700676924717144], [0.016107214752705958...","[[0.04571004199327887], [0.048363499148355145]...","[[0.8805903268259051], [0.8490836447298169], [..."
1,1,0.331385,SiebDesEratosthenes,"[[0.34468861682456864], [0.21735128639331527],...","[[0.09922513867470888], [0.12703114702627924],...","[[0.07170695707201084], [0.06795204534695756],...","[[0.18852976127026683], [0.27760773454710835],...","[[0.7115417014381397], [0.7121863298027544], [...","[[0.3791828346139767], [0.3505118526681873], [...","[[0.09764733023518274], [0.18989828695101543],...","[[0.06576426272198989], [0.07438454180738402],...","[[0.11486613203573147], [0.13952652057222267],...","[[0.6667103513924688], [0.7657983573085516], [..."
2,1,0.331385,IsAnagram,"[[0.15649327727940737], [0.11276743007249122],...","[[0.14900882300346255], [0.1695370237425774], ...","[[0.07183091222399918], [0.09332813125341706],...","[[0.13725417645852428], [0.204410920774691], [...","[[0.5477109775142269], [0.6046727972957898], [...","[[0.6600267837616767], [0.570125074824032], [0...","[[0.08022912244763095], [0.08285398602995175],...","[[0.04413988015041232], [0.03564129936749814],...","[[0.03029472956800593], [0.09235194051644119],...","[[0.8314566872242196], [0.8039002504877778], [..."
3,1,0.331385,RemoveDoubleChar,"[[0.34015158230076753], [0.20111192734298633],...","[[0.12451579945317702], [0.17081508193290285],...","[[0.07085129602393633], [0.08138112519264509],...","[[0.12472020035900913], [0.18165435455395615],...","[[0.6795079286809589], [0.6449188024942216], [...","[[0.7587329582865392], [0.6379710891783169], [...","[[0.06806638543924298], [0.09407397223434152],...","[[0.03790794626414092], [0.033204566798134195]...","[[0.04234483529875204], [0.09994236037744139],...","[[0.9102166406270168], [0.8810609640989394], [..."
4,1,0.331385,BinToDecimal,"[[0.274432316520621], [0.27320134944157615], [...","[[0.10702854590351592], [0.16555113663211815],...","[[0.061694315267520915], [0.08899926239143556]...","[[0.22841641648922786], [0.19047895483366234],...","[[0.695017562378288], [0.7351271385244439], [0...","[[0.33378639580805325], [0.234187918391723], [...","[[0.11710672627270548], [0.19381690148602812],...","[[0.056831697226970374], [0.09637173689065773]...","[[0.20633500540407507], [0.22659266836604416],...","[[0.725870464021275], [0.7648709950709858], [0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,DumpSorting,"[[0.1724965946827452], [0.1513985526692459], [...","[[0.15533138035278238], [0.12674567959003397],...","[[0.09774017555936307], [0.08165586980918603],...","[[0.10002327853580059], [0.07500799110168185],...","[[0.5426534730954017], [0.4464909493680884], [...","[[0.07714827723313789], [0.06157365782805608],...","[[0.17187699647340002], [0.1755360403526496], ...","[[0.07083044156725822], [0.0891231901747141], ...","[[0.05894912635430291], [0.07134160418274467],...","[[0.3869751678487524], [0.40778188154162853], ..."
1068,71,0.435651,BinomialCoefficient,"[[0.21640923365253614], [0.2200156163744832], ...","[[0.19148998256361835], [0.216900214586358], [...","[[0.06744739811806552], [0.06197044389155802],...","[[0.14999049708092205], [0.10047337134213218],...","[[0.640299251246129], [0.6040265132098765], [0...","[[0.20665167598019946], [0.14648226749279264],...","[[0.2563371203744964], [0.21277785765612747], ...","[[0.0615922728100755], [0.059321313291793736],...","[[0.0708845268544982], [0.0723816488796596], [...","[[0.600755

In [68]:
#Comapre between with and without baseline correction algorithm
df_with_baseline = df_brain_waves.copy()
df_without_baseline = pd.read_csv('../EEG_Result/Processing/BrainWaves.csv')
df_without_baseline = df_without_baseline[["Participant", "SkillScore", "Algorithm", "Alpha", "Beta", "Gamma", "Theta", "Range4to50Hz"]]

# Convert NumPy arrays to lists
df_with_baseline = df_with_baseline.applymap(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
df_without_baseline = df_without_baseline.applymap(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

df_difference = df_without_baseline.compare(df_with_baseline)
df_difference

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\1567433090.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_with_baseline = df_with_baseline.applymap(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\1567433090.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_without_baseline = df_without_baseline.applymap(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)


Alpha  \
                                                   self   
0     [[0.17390609]\n [0.15958725]\n [0.13287356]\n ...   
1     [[0.34468862]\n [0.21735129]\n [0.21592463]\n ...   
2     [[0.15649328]\n [0.11276743]\n [0.10716588]\n ...   
3     [[0.34015158]\n [0.20111193]\n [0.18510523]\n ...   
4     [[0.27443232]\n [0.27320135]\n [0.17806   ]\n ...   
...                                                 ...   
1067  [[0.17249659]\n [0.15139855]\n [0.06820992]\n ...   
1068  [[0.21640923]\n [0.22001562]\n [0.12874591]\n ...   
1069  [[0.12948521]\n [0.09379782]\n [0.05279457]\n ...   
1070  [[0.17840128]\n [0.2199462 ]\n [0.13685546]\n ...   
1071  [[0.14196692]\n [0.21579585]\n [0.09741747]\n ...   

                                                         \
                                                  other   
0     [[0.25115560656817276], [0.2471412199060335], ...   
1     [[0.25815663297896224], [0.1555127320397908], ...   
2     [[0.23880297184434107], [0.2855471691569154], ...   
3     [[0.3414489597538576], [0.34746052229390334], ...   
4     [[0.14888754832827367], [0.26849309528933063],...   
...                                                 ...   
1067  [[0.07489484843825855], [0.06642009427512763],...   
1068  [[0.1583508473460843], [0.15146752837954777], ...   
1069  [[0.18786224092885329], [0.1836114922505545], ...   
1070  [[0.26651103121208664], [0.22797499195567766],...   
1071  [[0.09135256743769571], [0.11401665746998974],...   

                                                   Beta  \
                                                   self   
0     [[0.0986916 ]\n [0.17572296]\n [0.19027423]\n ...   
1     [[0.09922514]\n [0.12703115]\n [0.14047499]\n ...   
2     [[0.14900882]\n [0.16953702]\n [0.21497959]\n ...   
3     [[0.1245158 ]\n [0.17081508]\n [0.18214232]\n ...   
4     [[0.10702855]\n [0.16555114]\n [0.14466075]\n ...   
...                                                 ...   
1067  [[0.15533138]\n [0.12674568]\n [0.09763525]\n ...   
1068  [[0.19148998]\n [0.21690021]\n [0.15069481]\n ...   
1069  [[0.22424443]\n [0.20794597]\n [0.18715376]\n ...   
1070  [[0.14574078]\n [0.19176457]\n [0.11012727]\n ...   
1071  [[0.25192871]\n [0.21269681]\n [0.16031805]\n ...   

                                                         \
                                                  other   
0     [[0.14263366632414762], [0.1979580673860483], ...   
1     [[0.09880985543251075], [0.15563668625028063],...   
2     [[0.10114077118132743], [0.1448721245386135], ...   
3     [[0.12016954239745795], [0.12627347977783812],...   
4     [[0.1035494386472099], [0.12842587657802085], ...   
...                                                 ...   
1067  [[0.1202695849329439], [0.12160176357047389], ...   
1068  [[0.1759501881140687], [0.15127822963385926], ...   
1069  [[0.13529448473120412], [0.1560159247188634], ...   
1070  [[0.1544990167397], [0.15326273473777613], [0....   
1071  [[0.1698487693640371], [0.15481333975338168], ...   

                                                  Gamma  \
                                                   self   
0     [[0.09502044]\n [0.07892027]\n [0.09148849]\n ...   
1     [[0.07170696]\n [0.06795205]\n [0.09560998]\n ...   
2     [[0.07183091]\n [0.09332813]\n [0.10819168]\n ...   
3     [[0.0708513 ]\n [0.08138113]\n [0.07831121]\n ...   
4     [[0.06169432]\n [0.08899926]\n [0.07380997]\n ...   
...                                                 ...   
1067  [[0.09774018]\n [0.08165587]\n [0.06502341]\n ...   
1068  [[0.0674474 ]\n [0.06197044]\n [0.08124911]\n ...   
1069  [[0.09659553]\n [0.09617712]\n [0.10361554]\n ...   
1070  [[0.0523235 ]\n [0.0581362 ]\n [0.09070209]\n ...   
1071  [[0.05830734]\n [0.09347557]\n [0.09308329]\n ...   

                                                         \
                                                  other   
0     [[0.08459141008787029], [0.08519439851261612],...   
1     [[0.10632342583330905], [0.

In [70]:
df_brain_waves_skilled = pd.merge(df_brain_waves, df_skilled[['Participant', 'SkillLevel']], on='Participant', how='left')
df_brain_waves_skilled = df_brain_waves_skilled[["Participant", "SkillScore", "SkillLevel", "Algorithm", "Alpha", "Beta", "Gamma", "Theta", "Range4to50Hz"]]
df_brain_waves_skilled.to_csv(result_path+"/BrainWavesSkilled.csv")
df_brain_waves_skilled

,Participant,SkillScore,SkillLevel,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz
0,1,0.331385,Intermediate,IsPrime,"[[0.25115560656817276], [0.2471412199060335], ...","[[0.14263366632414762], [0.1979580673860483], ...","[[0.08459141008787029], [0.08519439851261612],...","[[0.15915051689242432], [0.1606880897405415], ...","[[0.6618107147208608], [0.7133124831019936], [..."
1,1,0.331385,Intermediate,SiebDesEratosthenes,"[[0.25815663297896224], [0.1555127320397908], ...","[[0.09880985543251075], [0.15563668625028063],...","[[0.10632342583330905], [0.09601113308390134],...","[[0.2040251620691913], [0.1683366132221949], [...","[[0.6726010019298919], [0.5906614229188518], [..."
2,1,0.331385,Intermediate,IsAnagram,"[[0.23880297184434107], [0.2855471691569154], ...","[[0.10114077118132743], [0.1448721245386135], ...","[[0.0687197470400256], [0.05539180682255425], ...","[[0.14185770392942718], [0.18945928602106957],...","[[0.5585509165904352], [0.6944440350931635], [..."
3,1,0.331385,Intermediate,RemoveDoubleChar,"[[0.3414489597538576], [0.34746052229390334], ...","[[0.12016954239745795], [0.12627347977783812],...","[[0.055976387291174], [0.045340373933684924], ...","[[0.18540866673119105], [0.19483407518530282],...","[[0.7143030241286751], [0.7298783781613223], [..."
4,1,0.331385,Intermediate,BinToDecimal,"[[0.14888754832827367], [0.26849309528933063],...","[[0.1035494386472099], [0.12842587657802085], ...","[[0.08267931724040653], [0.07017483207470429],...","[[0.1981375467853718], [0.17694359353282613], ...","[[0.5368235421692543], [0.6480064384836258], [..."
...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,"[[0.07489484843825855], [0.06642009427512763],...","[[0.1202695849329439], [0.12160176357047389], ...","[[0.10736058068764368], [0.1099296380221142], ...","[[0.13380214588262995], [0.13666361572117924],...","[[0.46468635003508413], [0.45224666271855435],..."
1068,71,0.435651,Expert,BinomialCoefficient,"[[0.1583508473460843], [0.15146752837954777], ...","[[0.1759501881140687], [0.15127822963385926], ...","[[0.05255330414976637], [0.06798760435643422],...","[[0.1933608399911308], [0.18942581814465564], ...","[[0.5886176198556634], [0.5675787595199668], [..."
1069,71,0.435651,Expert,IsAnagram,"[[0.18786224092885329], [0.1836114922505545], ...","[[0.13529448473120412], [0.1560159247188634], ...","[[0.0776013233215809], [0.05327600600056234], ...","[[0.2463965566511211], [0.26483694608220343], ...","[[0.6800788378411011], [0.682907965100582], [0..."
1070,71,0.435651,Expert,ArrayAverage,"[[0.26651103121208664], [0.22797499195567766],...","[[0.1544990167397], [0.15326273473777613], [0....","[[0.04957322580429836], [0.06399188386211482],...","[[0.1920109144281394], [0.1981349145060469], [...","[[0.6778866223549568], [0.6618888883907834], [..."


In [71]:
df_brain_waves_skilled_unique = df_brain_waves_skilled[['Participant', 'SkillLevel']]
df_brain_waves_skilled_unique = df_brain_waves_skilled_unique.drop_duplicates()
df_brain_waves_skilled_unique.groupby('SkillLevel')['Participant'].count()

SkillLevel
Expert          13
Intermediate    12
Novice          12
Name: Participant, dtype: int64

In [72]:
# Statistical EEG features
def get_freqband_features(data):
    mean = np.mean(data)
    std_dev = np.std(data)
    skewness = skew(data)
    kurt = kurtosis(data)
    variance = np.var(data)
    median = np.median(data)    
    zero_cross_rate = len(find_peaks(data)[0]) / (len(data) - 1)
    entropy_data = entropy(data)
    
    return mean, std_dev, skewness, kurt, variance, median, zero_cross_rate, entropy_data

# Features

# 1. Based on Algorithm (ignoring electrode positions)

## 1.1. Brain Waves --> Alpha, Beta, Gamma, Theta waves

### 1.1.1 Code Comprehension Data

In [73]:
#get the code comprehension data brain waves features
df_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Algorithm', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves_skilled["Participant"].unique(), total=len(df_brain_waves_skilled["Participant"].unique())):
    df_participant = df_brain_waves_skilled[df_brain_waves_skilled["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["Alpha"]
        beta = row["Beta"]
        gamma = row["Gamma"]
        theta = row["Theta"]

        

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]

        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Algorithm': algorithm,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7],
        }, index=[0])

        df_brain_waves_stats = pd.concat([df_brain_waves_stats, algorithm_df], ignore_index=True)
        
df_brain_waves_stats.to_csv(result_path + "/features_alg_abgtbw_ccdata.csv")
df_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\1344662213.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_brain_waves_stats = pd.concat([df_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,IsPrime,0.238141,0.129326,0.070359,0.169713,0.080629,0.027704,...,0.071383,0.159779,0.354839,0.258065,0.290323,0.354839,3.407190,3.443719,3.436392,3.391787
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.245269,0.151492,0.063943,0.178115,0.077328,0.033770,...,0.062262,0.174744,0.290323,0.290323,0.354839,0.354839,3.418299,3.441761,3.443519,3.399024
2,1,0.331385,Intermediate,IsAnagram,0.196728,0.152572,0.068344,0.201643,0.081075,0.036006,...,0.068444,0.204620,0.354839,0.290323,0.290323,0.322581,3.383786,3.437370,3.425536,3.407341
3,1,0.331385,Intermediate,RemoveDoubleChar,0.162503,0.138889,0.096666,0.127092,0.060714,0.028008,...,0.097864,0.123381,0.258065,0.290323,0.258065,0.290323,3.397325,3.445755,3.427188,3.410328
4,1,0.331385,Intermediate,BinToDecimal,0.142000,0.129756,0.103908,0.107045,0.059809,0.030585,...,0.101619,0.104698,0.322581,0.258065,0.290323,0.290323,3.383445,3.439957,3.444137,3.403663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.173628,0.222050,0.077376,0.169103,0.088557,0.054914,...,0.061267,0.171863,0.258065,0.290323,0.258065,0.258065,3.348583,3.435289,3.353422,3.391227
1068,71,0.435651,Expert,BinomialCoefficient,0.198684,0.214792,0.053135,0.205762,0.093091,0.051732,...,0.042864,0.205898,0.322581,0.290323,0.322581,0.225806,3.368868,3.436928,3.370370,3.407139
1069,71,0.435651,Expert,IsAnagram,0.197533,0.213647,0.054574,0.204592,0.093086,0.052102,...,0.045760,0.211382,0.322581,0.290323,0.322581,0.258065,3.368974,3.436069,3.381901,3.407061
1070,71,0.435651,Expert,ArrayAverage,0.172305,0.219639,0.063944,0.189048,0.081006,0.047187,...,0.055809,0.177881,0.322581,0.290323,0.354839,0.290323,3.370779,3.442121,3.409815,3.416424


## 1.2 Brain Waves --> 4 to 50 Hz range 

In [75]:
df_brain_waves= df_brain_waves_skilled.copy()

### 1.2.1 Code Comprehension Data

In [76]:

df_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore','SkillLevel', 'Algorithm', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_freq_range = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        freq_range = row["Range4to50Hz"]
        merged_freq_range = np.concatenate((merged_freq_range, freq_range), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        channeled = merged_freq_range[idx, :]
       
        
        # Use the calculate_channel_statistics function on frequency band
        stats = get_freqband_features(channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Algorithm': algorithm,
            'Mean': stats[0],
            'StdDev': stats[1],
            'Skewness': stats[2],
            'Kurtosis': stats[3],
            'Variance': stats[4],
            'Median': stats[5],
            'ZeroCrossRate': stats[6],
            'Entropy': stats[7],
        }, index=[0])

        df_4to50Hz_brain_waves_stats = pd.concat([df_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)

df_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_alg_4to50hzbw_ccdata.csv')
df_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\2057372047.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_4to50Hz_brain_waves_stats = pd.concat([df_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Mean,StdDev,Skewness,Kurtosis,Variance,ZeroCrossRate,Entropy,Median
0,1,0.331385,Intermediate,IsPrime,0.623166,0.075672,-0.686589,0.378862,0.005726,0.354839,3.458087,0.640818
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.655634,0.073595,-0.364613,-0.554239,0.005416,0.322581,3.459315,0.666387
2,1,0.331385,Intermediate,IsAnagram,0.635358,0.074777,-0.381127,-0.654119,0.005592,0.354839,3.458666,0.652463
3,1,0.331385,Intermediate,RemoveDoubleChar,0.535681,0.068256,-0.891066,1.496665,0.004659,0.290323,3.457190,0.541280
4,1,0.331385,Intermediate,BinToDecimal,0.492292,0.063853,-0.575343,0.486091,0.004077,0.290323,3.457022,0.499086
...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.656499,0.103989,-0.635405,0.882013,0.010814,0.354839,3.452519,0.687090
1068,71,0.435651,Expert,BinomialCoefficient,0.686300,0.102730,-0.795942,1.710732,0.010553,0.354839,3.453835,0.704360
1069,71,0.435651,Expert,IsAnagram,0.684515,0.101655,-0.732439,1.596281,0.010334,0.354839,3.454073,0.695975
1070,71,0.435651,Expert,ArrayAverage,0.657239,0.093071,-0.224972,1.098488,0.008662,0.354839,3.455448,0.658380


# 2. Based on Electrode Position

In [77]:
df_brain_waves= df_brain_waves_skilled.copy()
df_brain_waves


,Participant,SkillScore,SkillLevel,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz
0,1,0.331385,Intermediate,IsPrime,"[[0.25115560656817276], [0.2471412199060335], ...","[[0.14263366632414762], [0.1979580673860483], ...","[[0.08459141008787029], [0.08519439851261612],...","[[0.15915051689242432], [0.1606880897405415], ...","[[0.6618107147208608], [0.7133124831019936], [..."
1,1,0.331385,Intermediate,SiebDesEratosthenes,"[[0.25815663297896224], [0.1555127320397908], ...","[[0.09880985543251075], [0.15563668625028063],...","[[0.10632342583330905], [0.09601113308390134],...","[[0.2040251620691913], [0.1683366132221949], [...","[[0.6726010019298919], [0.5906614229188518], [..."
2,1,0.331385,Intermediate,IsAnagram,"[[0.23880297184434107], [0.2855471691569154], ...","[[0.10114077118132743], [0.1448721245386135], ...","[[0.0687197470400256], [0.05539180682255425], ...","[[0.14185770392942718], [0.18945928602106957],...","[[0.5585509165904352], [0.6944440350931635], [..."
3,1,0.331385,Intermediate,RemoveDoubleChar,"[[0.3414489597538576], [0.34746052229390334], ...","[[0.12016954239745795], [0.12627347977783812],...","[[0.055976387291174], [0.045340373933684924], ...","[[0.18540866673119105], [0.19483407518530282],...","[[0.7143030241286751], [0.7298783781613223], [..."
4,1,0.331385,Intermediate,BinToDecimal,"[[0.14888754832827367], [0.26849309528933063],...","[[0.1035494386472099], [0.12842587657802085], ...","[[0.08267931724040653], [0.07017483207470429],...","[[0.1981375467853718], [0.17694359353282613], ...","[[0.5368235421692543], [0.6480064384836258], [..."
...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,"[[0.07489484843825855], [0.06642009427512763],...","[[0.1202695849329439], [0.12160176357047389], ...","[[0.10736058068764368], [0.1099296380221142], ...","[[0.13380214588262995], [0.13666361572117924],...","[[0.46468635003508413], [0.45224666271855435],..."
1068,71,0.435651,Expert,BinomialCoefficient,"[[0.1583508473460843], [0.15146752837954777], ...","[[0.1759501881140687], [0.15127822963385926], ...","[[0.05255330414976637], [0.06798760435643422],...","[[0.1933608399911308], [0.18942581814465564], ...","[[0.5886176198556634], [0.5675787595199668], [..."
1069,71,0.435651,Expert,IsAnagram,"[[0.18786224092885329], [0.1836114922505545], ...","[[0.13529448473120412], [0.1560159247188634], ...","[[0.0776013233215809], [0.05327600600056234], ...","[[0.2463965566511211], [0.26483694608220343], ...","[[0.6800788378411011], [0.682907965100582], [0..."
1070,71,0.435651,Expert,ArrayAverage,"[[0.26651103121208664], [0.22797499195567766],...","[[0.1544990167397], [0.15326273473777613], [0....","[[0.04957322580429836], [0.06399188386211482],...","[[0.1920109144281394], [0.1981349145060469], [...","[[0.6778866223549568], [0.6618888883907834], [..."


In [78]:
# read in one example file
example_cross_file = df_eeg_data.iloc[0]["CrossEEG"]
# read in the cross file
cross_eeg_data = mne.io.read_raw_fif(example_cross_file, verbose='ERROR')
# get the channel names
channel_names = cross_eeg_data.ch_names

## 2.1 Brain Waves --> Alpha, Beta, Gamma, Theta waves

### 2.1.1 Code Comprehension Data

In [79]:
#get features for code comprehension data
df_elec_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore','SkillLevel', 'Channel', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["Alpha"]
        beta = row["Beta"]
        gamma = row["Gamma"]
        theta = row["Theta"]

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
    # Calculate statistical features for each channel
    for idx, channel in enumerate(channel_names):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)

        
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7]
        }, index=[0])

        df_elec_brain_waves_stats = pd.concat([df_elec_brain_waves_stats, channel_df], ignore_index=True)

df_elec_brain_waves_stats.to_csv(result_path+'/features_elec_abgtbw_ccdata.csv')
df_elec_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\1908312530.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_brain_waves_stats = pd.concat([df_elec_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,Fp1,0.238141,0.129326,0.070359,0.169713,0.080629,0.027704,...,0.071383,0.159779,0.354839,0.258065,0.290323,0.354839,3.407190,3.443719,3.436392,3.391787
1,1,0.331385,Intermediate,Fp2,0.245269,0.151492,0.063943,0.178115,0.077328,0.033770,...,0.062262,0.174744,0.290323,0.290323,0.354839,0.354839,3.418299,3.441761,3.443519,3.399024
2,1,0.331385,Intermediate,F7,0.196728,0.152572,0.068344,0.201643,0.081075,0.036006,...,0.068444,0.204620,0.354839,0.290323,0.290323,0.322581,3.383786,3.437370,3.425536,3.407341
3,1,0.331385,Intermediate,F3,0.162503,0.138889,0.096666,0.127092,0.060714,0.028008,...,0.097864,0.123381,0.258065,0.290323,0.258065,0.290323,3.397325,3.445755,3.427188,3.410328
4,1,0.331385,Intermediate,Fz,0.142000,0.129756,0.103908,0.107045,0.059809,0.030585,...,0.101619,0.104698,0.322581,0.258065,0.290323,0.290323,3.383445,3.439957,3.444137,3.403663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.236272,0.240513,0.043285,0.197596,0.105429,0.062963,...,0.035990,0.198369,0.290323,0.354839,0.258065,0.290323,3.375199,3.432083,3.361661,3.409483
2364,71,0.435651,Expert,PO3,0.238771,0.222060,0.042111,0.196310,0.099607,0.047007,...,0.038188,0.175781,0.290323,0.322581,0.322581,0.258065,3.388639,3.442976,3.405187,3.410129
2365,71,0.435651,Expert,POz,0.227213,0.212462,0.043681,0.198680,0.097367,0.044738,...,0.038874,0.182642,0.322581,0.322581,0.258065,0.258065,3.382066,3.443291,3.406508,3.403189
2366,71,0.435651,Expert,PO4,0.210848,0.227646,0.046150,0.190215,0.082819,0.052666,...,0.041806,0.182340,0.354839,0.290323,0.290323,0.290323,3.396282,3.438058,3.416185,3.417107


## 2.2 Brain Waves --> 4 to 50 Hz range

### 2.2.1 Code Comprehension Data

In [80]:
#get features for code comprehension data
df_elec_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Channel', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'Median', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_full_spectrum = np.zeros(shape=(64, 0))
    
    for idx, row in df_participant.iterrows():
        full_spectrum = row["Range4to50Hz"]

        merged_full_spectrum = np.concatenate((merged_full_spectrum, full_spectrum), axis=1)

    for idx, channel in enumerate(channel_names):
        full_spectrum_channeled = merged_full_spectrum[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        full_spectrum_stats = get_freqband_features(full_spectrum_channeled)
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Mean': full_spectrum_stats[0],
            'StdDev': full_spectrum_stats[1],
            'Skewness': full_spectrum_stats[2],
            'Kurtosis': full_spectrum_stats[3],
            'Variance': full_spectrum_stats[4],
            'Median': full_spectrum_stats[5],
            'ZeroCrossRate': full_spectrum_stats[6],
            'Entropy': full_spectrum_stats[7],
        }, index=[0])

        df_elec_4to50Hz_brain_waves_stats = pd.concat([df_elec_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)

df_elec_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_elec_4to50hzbw_ccdata.csv')
df_elec_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_26520\995028288.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_4to50Hz_brain_waves_stats = pd.concat([df_elec_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Mean,StdDev,Skewness,Kurtosis,Variance,Median,ZeroCrossRate,Entropy
0,1,0.331385,Intermediate,Fp1,0.623166,0.075672,-0.686589,0.378862,0.005726,0.640818,0.354839,3.458087
1,1,0.331385,Intermediate,Fp2,0.655634,0.073595,-0.364613,-0.554239,0.005416,0.666387,0.322581,3.459315
2,1,0.331385,Intermediate,F7,0.635358,0.074777,-0.381127,-0.654119,0.005592,0.652463,0.354839,3.458666
3,1,0.331385,Intermediate,F3,0.535681,0.068256,-0.891066,1.496665,0.004659,0.541280,0.290323,3.457190
4,1,0.331385,Intermediate,Fz,0.492292,0.063853,-0.575343,0.486091,0.004077,0.499086,0.290323,3.457022
...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.734620,0.104349,-0.830146,1.763779,0.010889,0.743808,0.354839,3.455046
2364,71,0.435651,Expert,PO3,0.714518,0.099220,-1.036545,2.373916,0.009845,0.722794,0.322581,3.455412
2365,71,0.435651,Expert,POz,0.695990,0.104831,-0.803826,1.020181,0.010990,0.717473,0.354839,3.453713
2366,71,0.435651,Expert,PO4,0.689055,0.096053,-0.360006,0.421533,0.009226,0.693704,0.354839,3.455734


# Save the Result 

In [82]:
# List all files in the folder
files = os.listdir(result_path)

# Filter files starting with "features" and ending with ".csv"
feature_files = [file for file in files if file.startswith('features') and file.endswith('.csv')]

# Check if any matching files were found
if not feature_files:
    print("No matching CSV files found.")
else:
    # Initialize an empty list to store data
    df_processed = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','FilePath'])

    # Loop through each matching file
    for feature_file in feature_files:
        # Construct the full path to the CSV file
        csv_path = os.path.join(result_path, feature_file)

        # Extract information from the file name
        file_info = feature_file.split('_')
        focus = file_info[1]
        frequency_band = file_info[2].replace('bw', '') if len(file_info) > 2 else None
        datatype = file_info[3][:2] if len(file_info) > 3 else None

        #Clear Naming 
        if datatype == 'bl':
            datatype = 'Baseline'
        elif datatype == 'cc':
            datatype = 'CodeComprehension'

        if focus == 'alg':
            focus = 'Algorithm'
        elif focus == 'elec':
            focus = 'ElectrodePosition'

        if frequency_band == 'abgt':
            frequency_band = 'AlphaBetaThetaGamma'
        # Append data to the list
        df_processed.loc[len(df_processed)]= [focus, datatype, frequency_band, csv_path  ]

df_processed.to_csv(result_path + '/processed_data.csv')
df_processed

,Focus,Data,FrequencyBand,FilePath
0,Algorithm,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
1,Algorithm,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
2,ElectrodePosition,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
